In [1]:
# Imports
import torch
import pickle
import glob
import os
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

In [2]:
# Copy code from (my) github into the working directory, the code is slightly updated by me
# Without overflow bug 
!git clone https://github.com/moeschl21/TransformerLSR_moeschl.git
!cp -r TransformerLSR_moeschl/* /kaggle/working/
!rm -rf TransformerLSR_moeschl
!ls

Cloning into 'TransformerLSR_moeschl'...
remote: Enumerating objects: 266, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 266 (delta 56), reused 0 (delta 0), pack-reused 165 (from 1)
Receiving objects: 100% (266/266), 138.78 KiB | 2.43 MiB/s, done.
Resolving deltas: 100% (157/157), done.
brier.py	    functions.py  multiHeadAttention.py  TransformerLSR.py
data_simulation.py  loss.py	  README.md
eval.py		    main.py	  sublayers.py


In [3]:
# Create other directory for the output of the code
!mkdir -p results data work_dir data_lsr data_joinerml
!ls

brier.py       data_simulation.py  main.py		  sublayers.py
data	       eval.py		   multiHeadAttention.py  TransformerLSR.py
data_joinerml  functions.py	   README.md		  work_dir
data_lsr       loss.py		   results


In [4]:
# Set variables
num_traj = 100 # Number of patients
timeout = 1000 # Max visits
num_sim = 100

# Set missing
Y1_missing = 0.25
Y2_missing = 0.15
Y3_missing = 0.03

# Split the dataset
train_split = 0.6 # Note that the TransformerLSR needs to be adjusted for a different split within the code 
vali_split = 0.2
test_split = 0.2

In [5]:
# Data generation
for seed in range(1,num_sim+1): 
    print(f"##################### Seed: {seed} #####################")
    !python data_simulation.py \
        --seed {seed} \
        --num_traj {num_traj} \
        --timeout {timeout} \
        --save DIVAT_sim

    if seed == num_sim:
        print("done")

##################### Seed: 1 #####################
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
death rate: 0.34
mean death time: 593.6740384774732
mean visit: 5.3
mean time difference :151.92
Data time difference std:166.35
complete
##################### Seed: 2 #####################
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
censored
ce

In [6]:
# Split the dataset
for seed in range(1,num_sim+1):
    np.random.seed(seed)
    dataset_path = f'data/DIVAT_sim_{num_traj}_visit_{timeout}_long_3_seed_{seed}.pkl' 
    data_all = pd.read_pickle(dataset_path)
    data = data_all[data_all.obstime <= data_all.time]
    
    I = data["id"].values[-1]+1
    id_range = range(I)
    train_id = id_range[0:int(train_split*I)]
    vali_id = id_range[int(train_split*I):int((train_split+vali_split)*I)]
    test_id = id_range[int((train_split+vali_split)*I):I]

    train_data = data.loc[data["id"].isin(train_id)].copy()
    vali_data = data.loc[data["id"].isin(vali_id)].copy()
    test_data = data.loc[data["id"].isin(test_id)].copy()
    
    Y1_nan_inds = np.random.choice(train_data.index,size = int(Y1_missing*len(train_data)),replace=False)
    Y2_nan_inds = np.random.choice(train_data.index,size = int(Y2_missing*len(train_data)),replace=False)
    Y3_nan_inds = np.random.choice(train_data.index,size = int(Y3_missing*len(train_data)),replace=False)
    train_data.loc[Y1_nan_inds, "Y1"] = np.nan
    train_data.loc[Y2_nan_inds, "Y2"] = np.nan
    train_data.loc[Y3_nan_inds, "Y3"] = np.nan
    
    Y1_nan_inds_vali = np.random.choice(vali_data.index,size = int(Y1_missing*len(vali_data)),replace=False)
    Y2_nan_inds_vali = np.random.choice(vali_data.index,size = int(Y2_missing*len(vali_data)),replace=False)
    Y3_nan_inds_vali = np.random.choice(vali_data.index,size = int(Y3_missing*len(vali_data)),replace=False)
    vali_data.loc[Y1_nan_inds_vali, "Y1"] = np.nan
    vali_data.loc[Y2_nan_inds_vali, "Y2"] = np.nan
    vali_data.loc[Y3_nan_inds_vali, "Y3"] = np.nan
    
    Y1_nan_inds_test = np.random.choice(test_data.index,size = int(Y1_missing*len(test_data)),replace=False)
    Y2_nan_inds_test = np.random.choice(test_data.index,size = int(Y2_missing*len(test_data)),replace=False)
    Y3_nan_inds_test = np.random.choice(test_data.index,size = int(Y3_missing*len(test_data)),replace=False)
    test_data.loc[Y1_nan_inds_test, "Y1"] = np.nan
    test_data.loc[Y2_nan_inds_test, "Y2"] = np.nan
    test_data.loc[Y3_nan_inds_test, "Y3"] = np.nan


    
    data_LSR = pd.concat([train_data, vali_data, test_data],axis=0)
    output_path_LSR = (f"data_lsr/DIVAT_sim_missing_{num_traj}_visit_{timeout}_long_3_seed_{seed}.pkl")
    data_LSR.to_pickle(output_path_LSR)

    data_joineRML_train = pd.concat([train_data, vali_data],axis=0)
    data_joineRML_test = test_data
    output_path_joineRML_train = (f"data_joinerml/DIVAT_sim_missing_train_{num_traj}_visit_{timeout}_long_3_seed_{seed}.csv")
    output_path_joineRML_test = (f"data_joinerml/DIVAT_sim_missing_test_{num_traj}_visit_{timeout}_long_3_seed_{seed}.csv")
    data_joineRML_train.to_csv(output_path_joineRML_train,index=False)
    data_joineRML_test.to_csv(output_path_joineRML_test,index=False)

In [7]:
source = Path("/kaggle/working/data")
target = Path("/kaggle/working/data_lsr")

# Info-Datei kopieren und umbenennen
shutil.copy2(
    source / f"DIVAT_sim_{num_traj}_visit_{timeout}_long_3_info.pkl",
    target / f"DIVAT_sim_missing_{num_traj}_visit_{timeout}_long_3_info.pkl"
)

# PatientInfo-Dateien kopieren und umbenennen
for seed in range(1, num_sim + 1):
    shutil.copy2(
        source / f"DIVAT_sim_{num_traj}_visit_{timeout}_long_3_seed_{seed}_patientInfo.pkl",
        target / f"DIVAT_sim_missing_{num_traj}_visit_{timeout}_long_3_seed_{seed}_patientInfo.pkl"
    )

In [9]:
!zip -r data_joinerml.zip data_joinerml
!zip -r data_lsr.zip data_lsr

  adding: data_joinerml/ (stored 0%)
  adding: data_joinerml/DIVAT_sim_missing_test_100_visit_1000_long_3_seed_7.csv (deflated 68%)
  adding: data_joinerml/DIVAT_sim_missing_test_100_visit_1000_long_3_seed_84.csv (deflated 67%)
  adding: data_joinerml/DIVAT_sim_missing_test_100_visit_1000_long_3_seed_95.csv (deflated 67%)
  adding: data_joinerml/DIVAT_sim_missing_train_100_visit_1000_long_3_seed_2.csv (deflated 68%)
  adding: data_joinerml/DIVAT_sim_missing_train_100_visit_1000_long_3_seed_13.csv (deflated 69%)
  adding: data_joinerml/DIVAT_sim_missing_train_100_visit_1000_long_3_seed_96.csv (deflated 69%)
  adding: data_joinerml/DIVAT_sim_missing_test_100_visit_1000_long_3_seed_99.csv (deflated 68%)
  adding: data_joinerml/DIVAT_sim_missing_train_100_visit_1000_long_3_seed_66.csv (deflated 69%)
  adding: data_joinerml/DIVAT_sim_missing_train_100_visit_1000_long_3_seed_36.csv (deflated 68%)
  adding: data_joinerml/DIVAT_sim_missing_test_100_visit_1000_long_3_seed_69.csv (deflated 67%)


In [ ]:
!rm -rf data/*
!rm -rf data_joinerml/*
!rm -rf data_lsr/*
!rm data_joinerml.zip
!rm data_lsr.zip